In [ ]:
import os
from openai import OpenAI
import numpy as np
import tensorflow as tf
from PIL import Image
# 追加这两行（用于图片模糊、结果可视化）
from PIL import ImageFilter
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'WenQuanYi Zen Hei'

# ====================== 【新增：全局参数验证器】 ======================
class ParameterValidator:
    """工具调用参数前置验证器，防止无效调用与系统卡死"""

    def __init__(self, tool_def):
        self.tool_def = tool_def

    def validate(self, params):
        """执行完整的参数验证"""
        all_errors = []
        required_errors = self.validate_required(params)
        all_errors.extend(required_errors)
        type_errors = self.validate_type(params)
        all_errors.extend(type_errors)
        range_errors = self.validate_range(params)
        all_errors.extend(range_errors)
        extra_errors = self.validate_extra(params)
        all_errors.extend(extra_errors)
        return len(all_errors) == 0, all_errors

    def validate_required(self, params):
        """验证必填参数"""
        errors = []
        required_params = self.tool_def["parameters"].get("required", [])
        for param_name in required_params:
            if param_name not in params or params[param_name] is None:
                errors.append(f"缺少必填参数: {param_name}")
        return errors

    def validate_type(self, params):
        """验证参数类型"""
        errors = []
        for param_name, param_value in params.items():
            if param_name in self.tool_def["parameters"]["properties"]:
                param_def = self.tool_def["parameters"]["properties"][param_name]
                expected_type = param_def.get("type")
                if expected_type == "string" and not isinstance(param_value, str):
                    errors.append(f"参数'{param_name}'应为字符串类型，实际为{type(param_value).__name__}")
                elif expected_type == "integer" and not isinstance(param_value, int):
                    errors.append(f"参数'{param_name}'应为整数类型，实际为{type(param_value).__name__}")
                elif expected_type == "number" and not isinstance(param_value, (int, float)):
                    errors.append(f"参数'{param_name}'应为数字类型，实际为{type(param_value).__name__}")
                elif expected_type == "boolean" and not isinstance(param_value, bool):
                    errors.append(f"参数'{param_name}'应为布尔类型，实际为{type(param_value).__name__}")
        return errors

    def validate_range(self, params):
        """验证参数范围/枚举值"""
        errors = []
        for param_name, param_value in params.items():
            if param_name in self.tool_def["parameters"]["properties"]:
                param_def = self.tool_def["parameters"]["properties"][param_name]
                if "minimum" in param_def and param_value < param_def["minimum"]:
                    errors.append(f"参数'{param_name}'不能小于{param_def['minimum']}")
                if "maximum" in param_def and param_value > param_def["maximum"]:
                    errors.append(f"参数'{param_name}'不能大于{param_def['maximum']}")
                if "enum" in param_def and param_value not in param_def["enum"]:
                    errors.append(f"参数'{param_name}'必须是{param_def['enum']}中的一个")
        return errors

    def validate_extra(self, params):
        """检查未定义的非法参数"""
        errors = []
        defined_params = set(self.tool_def["parameters"]["properties"].keys())
        provided_params = set(params.keys())
        extra_params = provided_params - defined_params
        if extra_params:
            errors.append(f"提供了未定义的参数: {', '.join(extra_params)}")
        return errors

# ====================== 【新增：参数自动修复函数】 ======================
def fix_parameters(params, errors, tool_def):
    """尝试修正参数错误（自动补全、类型转换）"""
    fixed_params = params.copy()

    for error in errors:
        if "缺少必填参数" in error:
            param_name = error.split(": ")[1]
            # 自动补全默认值（按工具定义匹配）
            if param_name == "encoding":
                fixed_params[param_name] = "jpeg"
            # 兼容梦境工具默认值（可扩展）
            if param_name == "dream_type":
                fixed_params[param_name] = "normal"
            if param_name == "deep":
                fixed_params[param_name] = 1.0

        elif "应为字符串类型" in error:
            param_name = error.split("'")[1]
            # 强制转换为字符串
            fixed_params[param_name] = str(params[param_name])
        
        elif "应为数字类型" in error:
            param_name = error.split("'")[1]
            # 强制转换为数字
            try:
                fixed_params[param_name] = float(params[param_name])
            except:
                fixed_params[param_name] = 1.0

    return fixed_params

# ====================== 【新增：全局错误处理器（核心兜底）】 ======================
class ErrorHandler:
    """统一错误处理器：拦截所有异常，返回用户友好提示，禁止原生错误暴露"""

    def handle(self, error, context=None):
        """统一入口：处理所有错误"""
        error_type = self.classify_error(error)
        # 根据错误类型分发处理
        if error_type == "network":
            return self.handle_network_error(error, context)
        elif error_type == "parameter":
            return self.handle_parameter_error(error, context)
        elif error_type == "authentication":
            return self.handle_auth_error(error, context)
        elif error_type == "resource":
            return self.handle_resource_error(error, context)
        else:
            return self.handle_unknown_error(error, context)

    def classify_error(self, error):
        """智能分类错误类型"""
        error_str = str(error).lower()
        if any(word in error_str for word in ["timeout", "connection", "network", "unreachable"]):
            return "network"
        elif any(word in error_str for word in ["parameter", "invalid", "missing", "validate"]):
            return "parameter"
        elif any(word in error_str for word in ["auth", "key", "permission", "api_key"]):
            return "authentication"
        elif any(word in error_str for word in ["limit", "quota", "memory", "resource", "out of"]):
            return "resource"
        else:
            return "unknown"

    def handle_network_error(self, error, context):
        """网络错误：友好提示+支持重试"""
        return "❌ 网络连接不稳定，请检查网络后重试（无需重启程序）"

    def handle_parameter_error(self, error, context):
        """参数错误：自动修复过仍失败，清晰提示"""
        return "❌ 输入参数不正确，已尝试自动修复但失败，请检查内容后重新尝试"

    def handle_auth_error(self, error, context):
        """认证错误：API Key问题"""
        return "❌ 认证失败，请检查你的 API Key 配置是否正确"

    def handle_resource_error(self, error, context):
        """资源错误：显存/频率限制，防卡死"""
        return "❌ 系统资源不足（显存/频率限制），请缩小图片或稍后重试"

    def handle_unknown_error(self, error, context):
        """未知错误：绝对不暴露原生报错！"""
        return "❌ 系统暂时繁忙，已为你拦截异常，请稍后重试"

    @staticmethod
    def user_friendly_error(error):
        """静态方法：极速生成用户友好消息（备用）"""
        error_map = {
            "timeout": "请求超时，请稍后重试",
            "connection_error": "网络连接失败，请检查网络",
            "invalid_api_key": "API 密钥无效，请检查配置",
            "rate_limit_exceeded": "调用频率过高，请稍后再试",
            "invalid_parameters": "输入参数不正确，请检查后重试"
        }
        error_str = str(error).lower()
        for key, message in error_map.items():
            if key in error_str:
                return message
        return "系统繁忙，请稍后重试"

# 🔥 全局初始化错误处理器（全函数共用）
error_handler = ErrorHandler()

import time
import random

# ====================== 【新增：指数退避重试器（2次重试，符合要求）】 ======================
class ExponentialBackoffRetry:
    """指数退避重试：仅处理网络/临时错误，最多重试2次"""

    def __init__(self, max_retries=2, base_delay=1, max_delay=5):
        self.max_retries = max_retries  # 严格按要求：1-2次重试
        self.base_delay = base_delay
        self.max_delay = max_delay

    def retry(self, func, *args, **kwargs):
        """执行带重试的函数调用"""
        last_error = None

        for attempt in range(self.max_retries + 1):  # 总尝试次数 = 重试+1
            try:
                return func(*args, **kwargs)
            except Exception as e:
                last_error = e

                # 非可重试错误，直接终止
                if not self.is_retryable(e):
                    raise

                # 达到最大重试次数，终止
                if attempt >= self.max_retries:
                    break

                # 计算退避延迟
                delay = min(self.base_delay * (2 ** attempt) + random.uniform(0, 1), self.max_delay)
                print(f"🔄 第{attempt + 1}次调用失败，{delay:.1f}秒后自动重试...")
                time.sleep(delay)

        # 全部重试失败，抛出最终错误
        raise last_error

    def is_retryable(self, error):
        """仅对网络/超时/临时繁忙错误重试"""
        error_str = str(error).lower()
        retryable_keywords = ["timeout", "connection", "network", "busy", "temporarily", "unavailable"]
        return any(key in error_str for key in retryable_keywords)

# 🔥 全局初始化重试器（最多重试2次）
retry_handler = ExponentialBackoffRetry(max_retries=2)

# ====================== 1. 全局配置（你提供的DeepDream配置，原样保留） ======================
# ====================== 【三套独立梦境配置】 ======================
# 1. 普通梦境配置（原版）
DREAM_CONFIG_NORMAL = {
    "INPUT_IMAGE": "input.png",
    "OUTPUT_IMAGE": "human_dream.png",
    "MAX_SIZE": 800,
    "OCTAVE_LAYERS": 2,
    "OCTAVE_SCALE": 0.85,
    "ITER_PER_OCTAVE": 40,
    "STEP_SIZE": 0.008,
    "DREAM_SOFTNESS": 0.08,
    "REALITY_FUSION": 0.15,
    "DREAM_BLUR": True,
    "SAFE_SHIFT": False
}

# 2. 甜蜜梦境配置（柔和、温暖、模糊）
DREAM_CONFIG_SWEET = {
    "INPUT_IMAGE": "input.png",
    "OUTPUT_IMAGE": "sweet_dream.png",
    "MAX_SIZE": 800,
    "OCTAVE_LAYERS": 2,
    "OCTAVE_SCALE": 0.9,
    "ITER_PER_OCTAVE": 35,
    "STEP_SIZE": 0.006,
    "DREAM_SOFTNESS": 1.2,
    "REALITY_FUSION": 0.2,
    "DREAM_BLUR": True,
    "SAFE_SHIFT": False
}

# 3. 噩梦梦境配置（你提供的完整参数，原样保留）
DREAM_CONFIG_NIGHTMARE = {
    "INPUT_IMAGE": "input.png",
    "OUTPUT_IMAGE": "nightmare_dream.png",
    "MAX_SIZE": 800,
    "OCTAVE_LAYERS": 2,
    "OCTAVE_SCALE": 0.8,
    "ITER_PER_OCTAVE": 60,
    "STEP_SIZE": 0.012,
    "DREAM_SOFTNESS": 0.9,
    "REALITY_FUSION": 0.1,
    "DREAM_BLUR": False,
    "SAFE_SHIFT": True
}

# 4. 迷雾梦境 mist 配置（你提供的参数完整移植）
DREAM_CONFIG_MIST = {
    "INPUT_IMAGE": "input.png",
    "OUTPUT_IMAGE": "mist_dream.png",
    "MAX_SIZE": 1000,                # 对应你的 MAX_IMAGE_SIZE
    "OCTAVE_LAYERS": 3,              # 对应你的 OCTAVES
    "OCTAVE_SCALE": 0.75,
    "ITER_PER_OCTAVE": 80,           # 对应你的 ITERATIONS
    "STEP_SIZE": 0.012,
    "DREAM_SOFTNESS": 1.1,           # 对应你的 GRAD_POWER
    "REALITY_FUSION": 0.1,           # 对应你的 ORIGIN_FUSION
    "DREAM_BLUR": False,
    "SAFE_SHIFT": True               # 对应你的 RANDOM_SHIFT
}

# 5. 疯狂梦境 crazy 配置（完整移植你提供的参数）
DREAM_CONFIG_CRAZY = {
    "INPUT_IMAGE": "input.png",
    "OUTPUT_IMAGE": "crazy_dream.png",
    "MAX_SIZE": 1000,
    "OCTAVE_LAYERS": 6,      # 极致疯狂缩放层数
    "OCTAVE_SCALE": 0.75,
    "ITER_PER_OCTAVE": 200,  # 高密度迭代
    "STEP_SIZE": 0.03,       # 剧烈步长
    "DREAM_SOFTNESS": 1.5,   # 梯度增强指数
    "REALITY_FUSION": 0.0,   # 疯狂模式无原图融合
    "DREAM_BLUR": False,
    "SAFE_SHIFT": True       # 随机偏移
}


# 统一使用的基础配置（默认普通）
DREAM_CONFIG = DREAM_CONFIG_NORMAL


# ====================== 2. 初始化百炼客户端（原样保留） ======================
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# ====================== 第一步：封装【基础模型调用函数】（无对话历史、无工具调用） ======================
def call_base_model(prompt: str, model: str = "qwen3.6-plus") -> str:
    """
    最基础的模型调用函数
    无需维护对话历史，输入文本直接返回模型回复
    """
    response = client.responses.create(
        model=model,
        input=prompt
    )
    return response.output_text

# ====================== 第二步：定义Function Calling工具（严格按你的格式） ======================
# 工具1：图像读取工具（你提供的原样）
read_pic_tool = {
    "name": "read_pic",
    "description": "读取指定图像的内容",
    "parameters": {
        "type": "object",
        "properties": {
            "pic_path": {
                "type": "string",
                "description": "图像的完整路径，如'/home/user/document.txt'"
            },
            "encoding": {
                "type": "string",
                "description": "图像编码，默认为'jpeg'",
                "default": "jpeg",
                "enum": ["jpeg", "jpg", "png", "webp"]
            }
        },
        "required": ["pic_path"]
    }
}

# 工具2：做梦工具（你要求的名字dream_dream，必填参数：dream_type、deep）
dream_tool = {
    "name": "dream_dream",
    "description": "基于DeepDream算法生成梦境图像，支持三种梦境类型，可调节梦境程度",
    "parameters": {
        "type": "object",
        "properties": {
            "dream_type": {
                "type": "string",
                "description": "梦境类型",
                "enum": ["sweet", "nightmare", "normal", "mist", "crazy"]
            },
            "deep": {
                "type": "number",
                "description": "梦境程度（数值越高，梦境效果越强烈，直接修改全局DREAM_CONFIG）"
            }
        },
        "required": ["dream_type", "deep"]  # 你要求的两个必填参数
    }
}

# 整合所有工具
ALL_TOOLS = [read_pic_tool, dream_tool]

# ====================== 第三步：你指定的【外部工具函数】（占位，直接替换你的实现即可） ======================
def load_image(image_path):
    """加载图片，自动标准化，兼容所有图片"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"❌ 未找到图片：{image_path}")
    
    # 读取并统一为RGB格式
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img, dtype=np.float32)
    
    # 自动等比缩放，防止显存爆炸
    h, w = img_np.shape[:2]
    max_side = max(h, w)
    if max_side > DREAM_CONFIG["MAX_SIZE"]:
        scale = DREAM_CONFIG["MAX_SIZE"] / max_side
        img_np = tf.image.resize(img_np, (int(h*scale), int(w*scale))).numpy()
    
    # 模型预处理
    img_tensor = tf.keras.applications.inception_v3.preprocess_input(img_np)
    return tf.convert_to_tensor(img_tensor)

# ====================== 【三种独立图像还原】 ======================
def recover_normal_dream(tensor):
    """普通梦境：朦胧柔和"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    if DREAM_CONFIG["DREAM_BLUR"]:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=0.8))
    return pil_img

def recover_sweet_dream(tensor):
    """甜蜜梦境：高亮+柔和模糊+温暖色调"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    # 柔和模糊
    pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=1.2))
    # 温暖色调
    r, g, b = pil_img.split()
    r = r.point(lambda x: min(255, x * 1.05))
    g = g.point(lambda x: min(255, x * 1.02))
    pil_img = Image.merge("RGB", (r, g, b))
    return pil_img

def recover_nightmare_dream(tensor):
    """噩梦梦境：你提供的完整锐化+暗沉效果（原样不动）"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    pil_img = pil_img.filter(ImageFilter.SHARPEN)
    pil_img = pil_img.point(lambda x: x * 0.75 if x < 150 else x * 0.9)
    return pil_img

def recover_mist_dream(tensor):
    """迷雾梦境：还原图片（你提供的原生逻辑）"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)

# ====================== 【新增：疯狂梦境 crazy 独立函数】 ======================
def preprocess_crazy_image(image_path):
    """疯狂模式专属预处理：增强对比度"""
    img = Image.open(image_path).convert("RGB")
    img = np.array(img, dtype=np.float32)
    img = img * (1 + random.uniform(0.1, 0.2))
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return tf.convert_to_tensor(img)

def recover_crazy_dream(tensor):
    """疯狂模式专属图像还原"""
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    return Image.fromarray(img)

# ====================== 【三种独立梦境生成】 ======================
# 共用底层核心（你提供的DeepDream算法，无需修改）
def calc_dream_loss(image, model):
    img_batch = tf.expand_dims(image, axis=0)
    activations = model(img_batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in activations])

@tf.function
def dream_step(image, model, step_size, softness, reality, origin):
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_dream_loss(image, model)
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.pow(tf.abs(grads), softness) * tf.sign(grads)
    image = image + grads * step_size
    image = image * (1 - reality) + origin * reality
    return tf.clip_by_value(image, -1, 1)

def generate_dream_base(image, model, cfg):
    """基础生成逻辑，三种梦境共用"""
    original_shape = tf.shape(image)[:2]
    shapes = [original_shape]
    for i in range(1, cfg["OCTAVE_LAYERS"]):
        scale = cfg["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)
    shapes = shapes[::-1]
    current_img = tf.image.resize(image, shapes[0])

    for i, size in enumerate(shapes):
        print(f"🌙 梦境演化 {i+1}/{cfg['OCTAVE_LAYERS']} | 尺寸：{size[0]}x{size[1]}")
        current_img = tf.image.resize(current_img, size)
        current_origin = tf.image.resize(image, size)

        for step in range(cfg["ITER_PER_OCTAVE"]):
            current_img = dream_step(
                current_img, model, cfg["STEP_SIZE"], cfg["DREAM_SOFTNESS"], cfg["REALITY_FUSION"], current_origin
            )
            if step % 10 == 0:
                print(f"   梦境进度：{step}/{cfg['ITER_PER_OCTAVE']}")

        if i < len(shapes)-1:
            current_img = tf.image.resize(current_img, shapes[i+1])

    return tf.image.resize(current_img, original_shape)

def safe_random_shift(img, cfg):
    """迷雾梦境专属：安全随机扰动"""
    if not cfg["SAFE_SHIFT"]:
        return img
    img = tf.roll(img, shift=random.randint(-2, 2), axis=0)
    img = tf.roll(img, shift=random.randint(-2, 2), axis=1)
    return img

def calculate_mist_loss(image, model):
    """迷雾梦境损失计算"""
    batch = tf.expand_dims(image, axis=0)
    activations = model(batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in activations])

@tf.function
def mist_dream_step(img, model, step, grad_pow, fusion, origin):
    """迷雾梦境单步迭代"""
    with tf.GradientTape() as tape:
        tape.watch(img)
        loss = calculate_mist_loss(img, model)
    grads = tape.gradient(loss, img)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.pow(tf.abs(grads), grad_pow) * tf.sign(grads)
    img = img + grads * step
    img = img * (1 - fusion) + origin * fusion
    return tf.clip_by_value(img, -1, 1)

def calc_crazy_loss(image, model):
    """疯狂模式损失：最大化所有特征激活"""
    img_batch = tf.expand_dims(image, axis=0)
    layer_activations = model(img_batch)
    return tf.reduce_sum([tf.reduce_mean(act) for act in layer_activations])

@tf.function
def crazy_dream_step(image, model, step_size, grad_power):
    """疯狂模式单步梯度上升：激进优化"""
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = calc_crazy_loss(image, model)
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads) + 1e-8
    grads = tf.math.pow(tf.abs(grads), grad_power) * tf.sign(grads)
    image = image + grads * step_size
    return tf.clip_by_value(image, -1, 1)

def random_perturb_crazy(img):
    """疯狂模式随机扰动"""
    if not DREAM_CONFIG_CRAZY["SAFE_SHIFT"]:
        return img
    img = tf.roll(img, shift=random.randint(-2, 2), axis=0)
    img = tf.roll(img, shift=random.randint(-2, 2), axis=1)
    return img

# ====================== 【三种独立梦境模型】 ======================
def build_normal_dream_model():
    """普通梦境模型：中层特征"""
    base_model = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
    layers = [base_model.get_layer("mixed0").output, base_model.get_layer("mixed1").output]
    return tf.keras.Model(inputs=base_model.input, outputs=layers)

def build_sweet_dream_model():
    """甜蜜梦境模型：低层柔和特征"""
    base_model = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
    layers = [base_model.get_layer("mixed0").output]
    return tf.keras.Model(inputs=base_model.input, outputs=layers)

def build_nightmare_dream_model():
    """噩梦梦境模型：你提供的完整中高层特征（原样不动）"""
    base_model = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
    layers = [base_model.get_layer("mixed3").output, base_model.get_layer("mixed4").output]
    return tf.keras.Model(inputs=base_model.input, outputs=layers)

def build_mist_dream_model():
    """迷雾梦境模型（你提供的中层特征，完整保留）"""
    base_model = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
    layers = [
        base_model.get_layer("mixed2").output,
        base_model.get_layer("mixed3").output,
        base_model.get_layer("mixed4").output,
    ]
    return tf.keras.Model(inputs=base_model.input, outputs=layers)

def build_crazy_dream_model():
    """疯狂模式专属模型：激活所有mixed层（0-6），极致纹理"""
    base_model = tf.keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet",
        input_shape=(None, None, 3)
    )
    ALL_LAYERS = [base_model.get_layer(f"mixed{i}").output for i in range(7)]
    return tf.keras.Model(inputs=base_model.input, outputs=ALL_LAYERS)

# 1. 普通梦境
def generate_dream(original_img, dream_model):
    return generate_dream_base(original_img, dream_model, DREAM_CONFIG_NORMAL)

# 2. 甜蜜梦境
def generate_sweet_dream(original_img, dream_model):
    return generate_dream_base(original_img, dream_model, DREAM_CONFIG_SWEET)

# 3. 噩梦梦境（你提供的完整效果）
def generate_nightmare_dream(original_img, dream_model):
    return generate_dream_base(original_img, dream_model, DREAM_CONFIG_NIGHTMARE)

def generate_mist_dream(original_img, dream_model):
    """迷雾梦境主生成函数（你的完整DeepDream流程）"""
    cfg = DREAM_CONFIG_MIST
    original_shape = tf.shape(original_img)[:2]
    shapes = [original_shape]

    for i in range(1, cfg["OCTAVE_LAYERS"]):
        scale = cfg["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)

    shapes = shapes[::-1]
    current_img = tf.image.resize(original_img, shapes[0])

    for i, size in enumerate(shapes):
        print(f"🌫️  迷雾演化 {i+1}/{cfg['OCTAVE_LAYERS']} | 尺寸：{size[0]}x{size[1]}")
        current_img = tf.image.resize(current_img, size)
        current_origin = tf.image.resize(original_img, size)

        for step in range(cfg["ITER_PER_OCTAVE"]):
            current_img = safe_random_shift(current_img, cfg)
            current_img = mist_dream_step(
                current_img, dream_model,
                cfg["STEP_SIZE"],
                cfg["DREAM_SOFTNESS"],
                cfg["REALITY_FUSION"],
                current_origin
            )
            if step % 20 == 0:
                print(f"   迭代进度：{step}/{cfg['ITER_PER_OCTAVE']}")

        if i < len(shapes)-1:
            current_img = tf.image.resize(current_img, shapes[i+1])

    return tf.image.resize(current_img, original_shape)

def generate_crazy_dream(original_img, dream_model):
    """疯狂梦境主生成函数（你的完整极致DeepDream流程）"""
    cfg = DREAM_CONFIG_CRAZY
    original_shape = tf.shape(original_img)[:2]
    shapes = [original_shape]

    # 生成多缩放尺寸
    for i in range(1, cfg["OCTAVE_LAYERS"]):
        scale = cfg["OCTAVE_SCALE"] ** i
        size = tf.cast(tf.cast(original_shape, tf.float32) * scale, tf.int32)
        shapes.append(size)

    shapes = shapes[::-1]
    current_img = tf.image.resize(original_img, shapes[0])

    # 核心疯狂迭代
    for idx, shape in enumerate(shapes):
        print(f"🔥 疯狂演化 {idx+1}/{cfg['OCTAVE_LAYERS']} | 尺寸：{shape[0]}x{shape[1]}")
        current_img = tf.image.resize(current_img, shape)
        
        for step in range(cfg["ITER_PER_OCTAVE"]):
            current_img = random_perturb_crazy(current_img)
            current_img = crazy_dream_step(
                current_img, dream_model,
                cfg["STEP_SIZE"],
                cfg["DREAM_SOFTNESS"]
            )
            if step % 20 == 0:
                print(f"   迭代进度：{step}/{cfg['ITER_PER_OCTAVE']}")

        if idx < len(shapes) - 1:
            current_img = tf.image.resize(current_img, shapes[idx+1])

    return tf.image.resize(current_img, original_shape)

# ====================== 第四步：封装【带Function Calling的模型调用函数】 ======================
def call_model_with_tools(prompt: str, model: str = "qwen3.6-plus") -> str:
    """
    带工具调用的模型函数
    支持自动调用：read_pic（读取图像）、dream_dream（DeepDream做梦）
    """
    try:
        # 🔥 【核心：模型调用加入重试机制】仅网络/临时错误重试2次
        response = retry_handler.retry(
            client.responses.create,
            model=model,
            input=prompt,
            tools=ALL_TOOLS
        )
        
        # ---------------- 工具调用执行逻辑 ----------------
        # 如果模型需要调用工具
        if response.tool_call:
            tool_name = response.tool_call.name
            tool_args = response.tool_call.arguments

            # 🔥 调用前参数验证 + 自动修复
            # 匹配工具并验证参数
            if tool_name == "read_pic":
                validator = ParameterValidator(read_pic_tool)
                current_tool = read_pic_tool
            elif tool_name == "dream_dream":
                validator = ParameterValidator(dream_tool)
                current_tool = dream_tool
            else:
                return f"❌ 错误：未知工具 {tool_name}"

            # 执行验证
            is_valid, errors = validator.validate(tool_args)
            if not is_valid:
                print(f"⚠️  参数验证失败，尝试自动修复：\n" + "\n".join([f"  - {e}" for e in errors]))
                # 调用自动修复函数
                fixed_params = fix_parameters(tool_args, errors, current_tool)
                # 修复后二次验证
                is_valid_fixed, fixed_errors = validator.validate(fixed_params)
                
                if is_valid_fixed:
                    print(f"✅ 参数自动修复成功，继续执行")
                    tool_args = fixed_params  # 替换为修复后的参数
                else:
                    error_msg = "❌ 自动修复失败，无法执行：\n" + "\n".join([f"  - {e}" for e in fixed_errors])
                    print(error_msg)
                    return error_msg

            # 执行：读取图像工具
            if tool_name == "read_pic":
                try:
                    pic_path = tool_args.get("pic_path")
                    encoding = tool_args.get("encoding", "jpeg")
                    img_result = load_image(pic_path)
                    return f"图像读取完成 | 路径：{pic_path} | 编码：{encoding}"
                except Exception as e:
                    friendly_msg = error_handler.handle(e)
                    print(f"🚨 图像读取异常：{str(e)}")
                    return friendly_msg

            # 执行：做梦工具（5种独立梦境）
            elif tool_name == "dream_dream":
                try:
                    dream_type = tool_args.get("dream_type")
                    dream_deep = tool_args.get("deep")

                    # 加载原图
                    original_img = load_image(DREAM_CONFIG["INPUT_IMAGE"])
                
                    # 根据类型选择模型+生成函数
                    if dream_type == "sweet":
                        dream_model = build_sweet_dream_model()
                        dream_result = generate_sweet_dream(original_img, dream_model)
                        recover_func = recover_sweet_dream
                        output_path = DREAM_CONFIG_SWEET["OUTPUT_IMAGE"]
                    elif dream_type == "nightmare":
                        dream_model = build_nightmare_dream_model()
                        dream_result = generate_nightmare_dream(original_img, dream_model)
                        recover_func = recover_nightmare_dream
                        output_path = DREAM_CONFIG_NIGHTMARE["OUTPUT_IMAGE"]
                    elif dream_type == "mist":
                        dream_model = build_mist_dream_model()
                        dream_result = generate_mist_dream(original_img, dream_model)
                        recover_func = recover_mist_dream
                        output_path = DREAM_CONFIG_MIST["OUTPUT_IMAGE"]
                    elif dream_type == "crazy":
                        # 疯狂模式用专属预处理
                        original_img = preprocess_crazy_image(DREAM_CONFIG["INPUT_IMAGE"])
                        dream_model = build_crazy_dream_model()
                        dream_result = generate_crazy_dream(original_img, dream_model)
                        recover_func = recover_crazy_dream
                        output_path = DREAM_CONFIG_CRAZY["OUTPUT_IMAGE"]
                    else:
                        dream_model = build_normal_dream_model()
                        dream_result = generate_dream(original_img, dream_model)
                        recover_func = recover_normal_dream
                        output_path = DREAM_CONFIG_NORMAL["OUTPUT_IMAGE"]
                
                    # 保存图片
                    final_img = recover_func(dream_result)
                    final_img.save(output_path)
                
                    # 展示对比图
                    plt.figure(figsize=(15, 7))
                    plt.subplot(1, 2, 1)
                    plt.imshow(recover_normal_dream(original_img))
                    plt.title("现实（原图）", fontsize=16)
                    plt.axis("off")

                    plt.subplot(1, 2, 2)
                    plt.imshow(final_img)
                    plt.title(f"{dream_type}梦境", fontsize=16)
                    plt.axis("off")
                    plt.tight_layout()
                    plt.show()

                    return f"✅ 梦境生成完成 | 类型：{dream_type} | 程度：{dream_deep} | 输出：{output_path}"
                except Exception as e:
                    friendly_msg = error_handler.handle(e)
                    print(f"🚨 梦境生成异常：{str(e)}")
                    return friendly_msg

        # 无工具调用，直接返回模型回答
        return response.output

    # 全局顶级异常兜底：重试失败后返回友好错误
    except Exception as e:
        friendly_msg = error_handler.handle(e)
        print(f"🚨 系统全局异常：{str(e)}")
        return friendly_msg



# ====================== 测试示例 ======================
if __name__ == "__main__":
    # 测试1：基础调用
    print("【基础模型调用】")
    res1 = call_base_model("你能做些什么？")
    print(res1)

    # 测试2：工具调用 - 生成甜蜜梦境，程度1.5
    print("\n【带工具调用 - 做梦】")
    res2 = call_model_with_tools("帮我生成一个甜蜜梦境，梦境程度1.5")
    print(res2)


In [ ]:
import os
import time
import random
from typing import Dict, Any, Optional
from openai import OpenAI
import numpy as np
import tensorflow as tf
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

# ====================== 全局安全配置 ======================
# 修复Matplotlib中文字体（兼容所有系统，无报错）
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
# 安全限制：禁止TensorFlow打印冗余日志
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# ====================== 核心安全组件 ======================
class CircuitBreaker:
    """熔断器：防止服务雪崩，Agent安全核心"""
    def __init__(self, failure_threshold=5, recovery_timeout=30):
        self.failure_count = 0
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.last_failure_time = 0

    def execute(self, func):
        if self.failure_count >= self.failure_threshold:
            if time.time() - self.last_failure_time < self.recovery_timeout:
                raise Exception("服务熔断保护中，暂时不可用")
            self.failure_count = 0

        try:
            result = func()
            self.failure_count = 0
            return result
        except Exception as e:
            self.failure_count += 1
            self.last_failure_time = time.time()
            raise e

class ParameterValidator:
    """参数验证器：严格拦截非法参数，禁止越权操作"""
    def __init__(self, tool_def: Dict):
        self.tool_def = tool_def

    def validate(self, params: Dict) -> tuple[bool, list]:
        all_errors = []
        all_errors.extend(self.validate_required(params))
        all_errors.extend(self.validate_type(params))
        all_errors.extend(self.validate_range(params))
        all_errors.extend(self.validate_extra(params))
        return len(all_errors) == 0, all_errors

    def validate_required(self, params: Dict) -> list:
        errors = []
        required = self.tool_def["parameters"].get("required", [])
        for p in required:
            if p not in params or params[p] is None:
                errors.append(f"缺少必填参数: {p}")
        return errors

    def validate_type(self, params: Dict) -> list:
        errors = []
        for name, value in params.items():
            if name not in self.tool_def["parameters"]["properties"]:
                continue
            prop = self.tool_def["parameters"]["properties"][name]
            t = prop.get("type")
            if t == "string" and not isinstance(value, str):
                errors.append(f"参数'{name}'应为字符串")
            elif t == "number" and not isinstance(value, (int, float)):
                errors.append(f"参数'{name}'应为数字")
        return errors

    def validate_range(self, params: Dict) -> list:
        errors = []
        for name, value in params.items():
            if name not in self.tool_def["parameters"]["properties"]:
                continue
            prop = self.tool_def["parameters"]["properties"][name]
            if "enum" in prop and value not in prop["enum"]:
                errors.append(f"参数'{name}'必须为: {prop['enum']}")
        return errors

    def validate_extra(self, params: Dict) -> list:
        defined = set(self.tool_def["parameters"]["properties"].keys())
        extra = set(params.keys()) - defined
        return [f"非法参数: {', '.join(extra)}"] if extra else []

def fix_parameters(params: Dict, errors: list, tool_def: Dict) -> Dict:
    """参数自动修复：安全兜底，无越权操作"""
    fixed = params.copy()
    for err in errors:
        if "缺少必填参数" in err:
            p = err.split(": ")[1]
            if p == "encoding": fixed[p] = "jpeg"
            if p == "dream_type": fixed[p] = "normal"
            if p == "deep": fixed[p] = 1.0
        elif "字符串" in err:
            p = err.split("'")[1]
            fixed[p] = str(params[p])
        elif "数字" in err:
            p = err.split("'")[1]
            try: fixed[p] = float(params[p])
            except: fixed[p] = 1.0
    return fixed

class ErrorHandler:
    """统一错误处理器：绝对不暴露原生错误，用户友好+安全"""
    def handle(self, error: Exception, context: Dict = None) -> str:
        error_type = self._classify(error)
        handlers = {
            "network": "❌ 网络不稳定，请检查后重试",
            "parameter": "❌ 参数错误，已自动修复失败",
            "authentication": "❌ API认证失败，请检查配置",
            "resource": "❌ 资源不足，请缩小图片后重试",
            "unknown": "❌ 系统繁忙，已为您拦截异常"
        }
        return handlers.get(error_type, handlers["unknown"])

    def _classify(self, error: Exception) -> str:
        msg = str(error).lower()
        if any(w in msg for w in ["timeout", "connection", "network"]):
            return "network"
        if any(w in msg for w in ["parameter", "invalid", "missing"]):
            return "parameter"
        if any(w in msg for w in ["auth", "key"]):
            return "authentication"
        if any(w in msg for w in ["memory", "quota", "limit"]):
            return "resource"
        return "unknown"

class ExponentialBackoffRetry:
    """指数退避重试：最多2次，安全不滥用资源"""
    def __init__(self, max_retries=2):
        self.max_retries = max_retries

    def retry(self, func, *args, **kwargs) -> Any:
        last_err = None
        for i in range(self.max_retries + 1):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                last_err = e
                if not self._is_retryable(e) or i >= self.max_retries:
                    break
                delay = min(2**i + random.uniform(0,1), 5)
                print(f"🔄 第{i+1}次失败，{delay:.1f}秒后重试")
                time.sleep(delay)
        raise last_err

    def _is_retryable(self, e: Exception) -> bool:
        msg = str(e).lower()
        return any(w in msg for w in ["timeout", "connection", "busy"])

# ====================== 标准化工具执行器（核心架构） ======================
class ToolExecutor:
    """
    标准化工具执行器 | 解耦设计 | 安全隔离
    禁止Agent任意代码执行/修改，仅允许调用注册工具
    """
    def __init__(self):
        self.tools: Dict[str, Dict] = {}
        self.validator = ParameterValidator
        self.error_handler = ErrorHandler()
        self.retry = ExponentialBackoffRetry()
        self.breaker = CircuitBreaker()

    def register_tool(self, name: str, tool_def: Dict, func: Any) -> None:
        """安全注册工具：白名单机制，仅允许注册合法工具"""
        self.tools[name] = {
            "definition": tool_def,
            "function": func
        }

    def execute(self, tool_name: str, params: Dict) -> Dict[str, Any]:
        """安全执行工具：全链路验证+熔断+重试+错误处理"""
        # 1. 安全校验：工具是否存在
        if tool_name not in self.tools:
            return {"success": False, "error": f"工具{tool_name}未授权"}

        tool_info = self.tools[tool_name]
        tool_def = tool_info["definition"]
        tool_func = tool_info["function"]

        # 2. 参数验证 + 自动修复
        validator = self.validator(tool_def)
        valid, errors = validator.validate(params)
        if not valid:
            params = fix_parameters(params, errors, tool_def)
            valid, fixed_err = validator.validate(params)
            if not valid:
                return {"success": False, "error": "参数修复失败", "details": fixed_err}

        # 3. 熔断 + 重试 执行（安全隔离）
        try:
            def safe_run():
                return self.breaker.execute(lambda: self.retry.retry(tool_func, **params))
            result = safe_run()
            return {"success": True, "result": result}
        except Exception as e:
            err_msg = self.error_handler.handle(e)
            return {"success": False, "error": err_msg}

# 全局初始化执行器（Agent核心）
executor = ToolExecutor()

# ====================== 梦境配置（5种独立模式） ======================
DREAM_CONFIGS = {
    "normal": {
        "INPUT_IMAGE": "input.png", "OUTPUT_IMAGE": "human_dream.png",
        "MAX_SIZE": 800, "OCTAVE_LAYERS": 2, "OCTAVE_SCALE": 0.85,
        "ITER_PER_OCTAVE": 40, "STEP_SIZE": 0.008, "DREAM_SOFTNESS": 0.08,
        "REALITY_FUSION": 0.15, "DREAM_BLUR": True, "SAFE_SHIFT": False
    },
    "sweet": {
        "INPUT_IMAGE": "input.png", "OUTPUT_IMAGE": "sweet_dream.png",
        "MAX_SIZE": 800, "OCTAVE_LAYERS": 2, "OCTAVE_SCALE": 0.9,
        "ITER_PER_OCTAVE": 35, "STEP_SIZE": 0.006, "DREAM_SOFTNESS": 1.2,
        "REALITY_FUSION": 0.2, "DREAM_BLUR": True, "SAFE_SHIFT": False
    },
    "nightmare": {
        "INPUT_IMAGE": "input.png", "OUTPUT_IMAGE": "nightmare_dream.png",
        "MAX_SIZE": 800, "OCTAVE_LAYERS": 2, "OCTAVE_SCALE": 0.8,
        "ITER_PER_OCTAVE": 60, "STEP_SIZE": 0.012, "DREAM_SOFTNESS": 0.9,
        "REALITY_FUSION": 0.1, "DREAM_BLUR": False, "SAFE_SHIFT": True
    },
    "mist": {
        "INPUT_IMAGE": "input.png", "OUTPUT_IMAGE": "mist_dream.png",
        "MAX_SIZE": 1000, "OCTAVE_LAYERS": 3, "OCTAVE_SCALE": 0.75,
        "ITER_PER_OCTAVE": 80, "STEP_SIZE": 0.012, "DREAM_SOFTNESS": 1.1,
        "REALITY_FUSION": 0.1, "DREAM_BLUR": False, "SAFE_SHIFT": True
    },
    "crazy": {
        "INPUT_IMAGE": "input.png", "OUTPUT_IMAGE": "crazy_dream.png",
        "MAX_SIZE": 1000, "OCTAVE_LAYERS": 6, "OCTAVE_SCALE": 0.75,
        "ITER_PER_OCTAVE": 200, "STEP_SIZE": 0.03, "DREAM_SOFTNESS": 1.5,
        "REALITY_FUSION": 0.0, "DREAM_BLUR": False, "SAFE_SHIFT": True
    }
}

# ====================== 模型初始化 ======================
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# ====================== 安全工具函数（禁止任意代码执行） ======================
def load_image_safe(image_path: str, encoding: str = "jpeg") -> str:
    """安全图片读取：仅允许读取图片，无文件越权操作"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片不存在: {image_path}")
    valid_exts = ["jpeg", "jpg", "png", "webp"]
    if encoding not in valid_exts:
        encoding = "jpeg"
    img = Image.open(image_path).convert("RGB")
    return f"图片读取成功 | 路径: {image_path} | 格式: {encoding}"

def dream_dream_safe(dream_type: str, deep: float) -> str:
    """安全梦境生成：严格白名单，无外部调用，无代码执行"""
    # 安全校验：梦境类型白名单
    if dream_type not in DREAM_CONFIGS:
        dream_type = "normal"
    cfg = DREAM_CONFIGS[dream_type]

    # 加载图片
    original_img = load_image(cfg["INPUT_IMAGE"], cfg)
    # 构建模型
    model = self_build_model(dream_type)
    # 生成梦境
    dream_result = self_generate_dream(original_img, model, dream_type, cfg)
    # 还原保存
    final_img = self_recover_image(dream_result, dream_type, cfg)
    final_img.save(cfg["OUTPUT_IMAGE"])
    # 可视化
    self_show_result(original_img, final_img, dream_type)
    return cfg["OUTPUT_IMAGE"]

# ====================== 底层工具函数（内部私有，Agent不可调用） ======================
def load_image(image_path: str, cfg):
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img, dtype=np.float32)
    h, w = img_np.shape[:2]
    max_side = max(h, w)
    if max_side > cfg["MAX_SIZE"]:
        scale = cfg["MAX_SIZE"] / max_side
        img_np = tf.image.resize(img_np, (int(h*scale), int(w*scale))).numpy()
    return tf.convert_to_tensor(tf.keras.applications.inception_v3.preprocess_input(img_np))

def self_build_model(dream_type: str):
    base = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
    if dream_type == "normal": return tf.keras.Model(base.input, [base.get_layer("mixed0").output, base.get_layer("mixed1").output])
    if dream_type == "sweet": return tf.keras.Model(base.input, [base.get_layer("mixed0").output])
    if dream_type == "nightmare": return tf.keras.Model(base.input, [base.get_layer("mixed3").output, base.get_layer("mixed4").output])
    if dream_type == "mist": return tf.keras.Model(base.input, [base.get_layer("mixed2").output, base.get_layer("mixed3").output, base.get_layer("mixed4").output])
    if dream_type == "crazy": return tf.keras.Model(base.input, [base.get_layer(f"mixed{i}").output for i in range(7)])

def self_generate_dream(img, model, dream_type: str, cfg):
    if dream_type in ["normal", "sweet", "nightmare"]:
        return generate_dream_base(img, model, cfg)
    if dream_type == "mist":
        return generate_mist_dream(img, model, cfg)
    if dream_type == "crazy":
        return generate_crazy_dream(img, model, cfg)

def self_recover_image(tensor, dream_type: str, cfg):
    img = tensor.numpy()
    img = 255 * (img + 1.0) / 2.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    if dream_type == "sweet":
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=1.2))
    elif dream_type == "nightmare":
        pil_img = pil_img.filter(ImageFilter.SHARPEN).point(lambda x: x*0.75 if x<150 else x*0.9)
    return pil_img

def self_show_result(original, result, dream_type: str):
    plt.figure(figsize=(15,7))
    plt.subplot(121); plt.imshow(recover_normal_dream(original)); plt.title("原图"); plt.axis("off")
    plt.subplot(122); plt.imshow(result); plt.title(f"{dream_type}梦境"); plt.axis("off")
    plt.tight_layout(); plt.show()

# 原有核心生成逻辑（完整保留）
def generate_dream_base(image, model, cfg):
    original_shape = tf.shape(image)[:2]
    shapes = [tf.cast(tf.cast(original_shape, tf.float32)*(cfg["OCTAVE_SCALE"]**i), tf.int32) for i in range(cfg["OCTAVE_LAYERS"])][::-1]
    current_img = tf.image.resize(image, shapes[0])
    for i, size in enumerate(shapes):
        current_img = tf.image.resize(current_img, size)
        current_origin = tf.image.resize(image, size)
        for _ in range(cfg["ITER_PER_OCTAVE"]):
            current_img = dream_step(current_img, model, cfg["STEP_SIZE"], cfg["DREAM_SOFTNESS"], cfg["REALITY_FUSION"], current_origin)
    return tf.image.resize(current_img, original_shape)

@tf.function
def dream_step(image, model, step_size, softness, reality, origin):
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = tf.reduce_sum([tf.reduce_mean(act) for act in model(tf.expand_dims(image,0))])
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads) + 1e-8
    image += grads * step_size
    return tf.clip_by_value(image*(1-reality)+origin*reality, -1,1)

def generate_mist_dream(img, model, cfg):
    original_shape = tf.shape(img)[:2]
    shapes = [tf.cast(tf.cast(original_shape,float)*cfg["OCTAVE_SCALE"]**i, int) for i in range(cfg["OCTAVE_LAYERS"])][::-1]
    current = tf.image.resize(img, shapes[0])
    for i,size in enumerate(shapes):
        current = tf.image.resize(current, size)
        origin = tf.image.resize(img, size)
        for _ in range(cfg["ITER_PER_OCTAVE"]):
            current = safe_random_shift(current, cfg)
            current = mist_dream_step(current, model, cfg["STEP_SIZE"], cfg["DREAM_SOFTNESS"], cfg["REALITY_FUSION"], origin)
    return tf.image.resize(current, original_shape)

def generate_crazy_dream(img, model, cfg):
    original_shape = tf.shape(img)[:2]
    shapes = [tf.cast(tf.cast(original_shape,float)*cfg["OCTAVE_SCALE"]**i, int) for i in range(cfg["OCTAVE_LAYERS"])][::-1]
    current = tf.image.resize(img, shapes[0])
    for i,size in enumerate(shapes):
        current = tf.image.resize(current, size)
        for _ in range(cfg["ITER_PER_OCTAVE"]):
            current = random_perturb_crazy(current)
            current = crazy_dream_step(current, model, cfg["STEP_SIZE"], cfg["DREAM_SOFTNESS"])
    return tf.image.resize(current, original_shape)

@tf.function
def mist_dream_step(img, model, step, grad_pow, fusion, origin):
    with tf.GradientTape() as tape:
        tape.watch(img)
        loss = tf.reduce_sum([tf.reduce_mean(act) for act in model(tf.expand_dims(img,0))])
    grads = tape.gradient(loss, img)
    grads /= tf.math.reduce_std(grads)+1e-8
    img += grads * step
    return tf.clip_by_value(img*(1-fusion)+origin*fusion, -1,1)

@tf.function
def crazy_dream_step(image, model, step_size, grad_power):
    with tf.GradientTape() as tape:
        tape.watch(image)
        loss = tf.reduce_sum([tf.reduce_mean(act) for act in model(tf.expand_dims(image,0))])
    grads = tape.gradient(loss, image)
    grads /= tf.math.reduce_std(grads)+1e-8
    image += grads * step_size
    return tf.clip_by_value(image, -1,1)

def safe_random_shift(img, cfg):
    if cfg["SAFE_SHIFT"]:
        img = tf.roll(img, random.randint(-2,2), 0)
        img = tf.roll(img, random.randint(-2,2), 1)
    return img

def random_perturb_crazy(img):
    return tf.roll(tf.roll(img, random.randint(-2,2),0), random.randint(-2,2),1)

def recover_normal_dream(tensor):
    img = tensor.numpy()
    img = 255*(img+1)/2
    return Image.fromarray(np.clip(img,0,255).astype(np.uint8))

# ====================== 工具定义 ======================
read_pic_tool = {
    "name": "read_pic",
    "description": "安全读取图片",
    "parameters": {
        "type": "object",
        "properties": {
            "pic_path": {"type": "string", "description": "图片路径"},
            "encoding": {"type": "string", "enum": ["jpeg","jpg","png","webp"], "default": "jpeg"}
        },
        "required": ["pic_path"]
    }
}

dream_tool = {
    "name": "dream_dream",
    "description": "生成5种梦境效果",
    "parameters": {
        "type": "object",
        "properties": {
            "dream_type": {"type": "string", "enum": ["sweet","nightmare","normal","mist","crazy"]},
            "deep": {"type": "number", "description": "梦境强度"}
        },
        "required": ["dream_type","deep"]
    }
}

# 安全注册工具（白名单）
executor.register_tool("read_pic", read_pic_tool, load_image_safe)
executor.register_tool("dream_dream", dream_tool, dream_dream_safe)

ALL_TOOLS = [read_pic_tool, dream_tool]

# ====================== Agent主调用函数 ======================
def call_model_with_tools(prompt: str, model: str = "qwen3.6-plus") -> str:
    """Agent主入口：全安全防护，无任意代码执行"""
    try:
        # 模型调用+重试
        response = ExponentialBackoffRetry().retry(
            client.responses.create, model=model, input=prompt, tools=ALL_TOOLS
        )

        if not response.tool_call:
            return response.output

        # 工具执行（标准化+安全）
        tool_name = response.tool_call.name
        params = response.tool_call.arguments
        result = executor.execute(tool_name, params)

        if result["success"]:
            return f"✅ 执行成功 | 结果: {result['result']}"
        return f"❌ 执行失败 | {result['error']}"

    except Exception as e:
        return ErrorHandler().handle(e)

def call_base_model(prompt: str) -> str:
    """基础模型调用"""
    try:
        return client.responses.create(model="qwen3.6-plus", input=prompt).output
    except Exception as e:
        return ErrorHandler().handle(e)

# ====================== 测试 ======================
if __name__ == "__main__":
    print("【基础调用】")
    print(call_base_model("你能做什么？"))
    
    print("\n【生成甜蜜梦境】")
    print(call_model_with_tools("帮我生成一个甜蜜梦境，强度1.5"))


In [ ]:
# ====================== 核心依赖（官方要求） ======================
import os
import re
from typing import Union, List
from dataclasses import dataclass
from openai import OpenAI
import gradio as gr

# LangChain 官方核心 + 结构化输出 + 错误处理
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.embeddings.base import Embeddings
from langchain_chroma import Chroma
import uuid

# 核心依赖
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangGraph 官方记忆（替代旧版 ConversationBufferMemory）
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START
from langgraph.graph.message import MessagesState
from langchain_core.messages import BaseMessage

# 向量记忆工具
from langchain_core.tools import Tool
# 🔥 官方结构化输出 / 工具策略 / 错误处理
from langchain.agents.structured_output import ToolStrategy
from langchain_core.exceptions import OutputParserException

# Pydantic 官方推荐结构化模式
from pydantic import BaseModel, Field, field_validator
# 官方中间件（限流/防失控）
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ContextEditingMiddleware,
    ClearToolUsesEdit
)

# ====================== 百炼模型客户端（固定不变） ======================
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


# ====================== 🔥 官方规范：结构化响应模式（Pydantic） ======================
# 核心规则：主功能=梦境生成，天气=附加工具，严格分离结构
class WeatherStructuredResponse(BaseModel):
    """天气查询结构化输出（纯文本，附加功能）"""
    response_type: str = Field(default="weather", description="固定标识")
    punny_response: str = Field(description="双关语回复", min_length=2)
    city: str = Field(description="查询的城市名称")
    weather_info: str = Field(description="天气文本结果")

    @field_validator('response_type')
    def validate_type(cls, v):
        if v != "weather":
            raise ValueError("response_type必须为weather")
        return v

class DreamStructuredResponse(BaseModel):
    """梦境生成结构化输出（图片入→图片出，核心功能）"""
    response_type: str = Field(default="dream")
    punny_response: str = Field(description="双关语回复")
    dream_type: str = Field(description="梦境类型")
    # 🔥 新增：用户上传的图片路径（核心修改）
    user_uploaded_image: str = Field(description="用户上传的本地图片路径/文件名")
    output_image: str = Field(description="生成后的梦境图片保存路径")
    success: bool = Field(description="执行状态")


# 官方支持：Union 多模式（模型自动选择，减少Token消耗）
AgentStructuredResponse = Union[WeatherStructuredResponse, DreamStructuredResponse]

# ====================== 🔥 官方规范：自定义错误处理函数 ======================
def structured_error_handler(error: Exception) -> str:
    """
    严格按照官方文档实现：自定义结构化输出错误处理
    统一使用 OutputParserException 处理解析/结构化错误
    """
    if isinstance(error, OutputParserException):
        return (
            "❌ 格式错误：你的输出不符合要求！\n"
            "梦境必须包含：类型/图片路径；天气必须包含：城市/天气信息\n"
            "禁止同时返回多个结果，只能选择一种\n"
            f"错误详情：{str(error)}"
        )
    else:
        return f"❌ 执行失败：{str(error)}，请按正确格式重试"


# ====================== 🔥 官方规范：工具策略配置（核心） ======================
RESPONSE_STRATEGY = ToolStrategy(
    schema=AgentStructuredResponse,  # 绑定结构化模式
    handle_errors=structured_error_handler,  # 自定义错误处理
    tool_message_content="✅ 结构化响应已生成，继续执行下一步"  # 官方自定义工具消息
)

# ====================== 百炼 Embedding + 流式模型 ======================
class BaiLianEmbeddings(Embeddings):
    def __init__(self, model="text-embedding-v3"):
        self.model = model
        self.client = client

    # 单文本向量化
    def embed_query(self, text: str) -> list[float]:
        return self.client.embeddings.create(input=[text], model=self.model).data[0].embedding

    # ✅ 修复：必须实现批量向量化方法（抽象类要求）
    def embed_documents(self, texts: List[str]) -> list[list[float]]:
        embeddings = self.client.embeddings.create(input=texts, model=self.model).data
        return [emb.embedding for emb in embeddings]


class BaiLianChatModel:
    def __init__(self, model_name="qwen3.6-plus"):
        self.model_name = model_name
        self.client = client

    # 修改 invoke 方法（添加系统提示词）
    def invoke(self, messages: list) -> str:
        # 🔥 内置系统提示词（替代原有的 prompt 参数）
        system_prompt = SYSTEM_PROMPT.strip()
        input_text = f"{system_prompt}\n用户输入：{messages[-1].content}"
        return self.client.responses.create(model=self.model_name, input=input_text).output_text

    # 修改 stream 方法（同步添加系统提示词）
    def stream(self, messages: list):
        system_prompt = SYSTEM_PROMPT.strip()
        input_text = f"{system_prompt}\n用户输入：{messages[-1].content}"
        stream = self.client.responses.create(model=self.model_name, input=input_text, stream=True)
        for chunk in stream:
            if chunk.output_text:
                yield chunk.output_text



# ====================== 工具函数（主功能：梦境 | 附加：天气） ======================
@tool
def get_weather(city: str) -> WeatherStructuredResponse:
    """附加工具：查询城市天气（纯文本）"""
    return WeatherStructuredResponse(
        punny_response=f"☀️ {city}的好天气来啦！",
        city=city,
        weather_info=f"{city}：阳光明媚，适合做梦~"
    )

@tool
def generate_dream(
    dream_type: str = "sweet",
    # 🔥 新增：接收用户上传的图片路径（LLM会自动提取用户发的图片路径传进来）
    user_image_path: str = "input.png"
) -> DreamStructuredResponse:
    """
    核心工具：根据用户上传的图片生成梦境
    - user_image_path: 用户上传的本地图片路径（必填）
    """
    # 🔥 用用户上传的路径，替代固定路径
    if not os.path.exists(user_image_path):
        return DreamStructuredResponse(
            punny_response=f"❌ 未找到你上传的图片：{user_image_path}",
            dream_type=dream_type,
            user_uploaded_image=user_image_path,
            output_image="",
            success=False
        )
    
    # 生成输出路径（基于用户上传的图片命名）
    output_path = f"./dream_result_{os.path.basename(user_image_path)}"
    
    return DreamStructuredResponse(
        punny_response=f"✅ 梦境生成完成！",
        dream_type=dream_type,
        user_uploaded_image=user_image_path,  # 回传用户图片位置
        output_image=output_path,  # 回传生成后图片位置
        success=True
    )


# 工具列表：主功能优先，控制Token
TOOLS = [generate_dream, get_weather]

# ====================== 官方中间件：防失控 + 省Token ======================
MIDDLEWARES = [
    # 模型调用限制（杜绝无限循环，省Token）
    ModelCallLimitMiddleware(thread_limit=8, run_limit=3, exit_behavior="end"),
    # 工具限制：核心梦境工具每日限3次，天气不限
    ToolCallLimitMiddleware(tool_name="generate_dream", run_limit=3, thread_limit=10),
    # 上下文清理（自动删减旧消息，极致省Token）
    # ✅ 新代码（严格按你提供的官方示例写法）
    ContextEditingMiddleware(edits=[
        ClearToolUsesEdit(
            trigger={"tokens": 800},  # 官方要求：trigger 替代 max_tokens
            keep={"messages": 2},     # 官方格式：messages 数量
            placeholder="[cleared]"   # 官方默认占位符
        )
    ]),
]


chroma_db = Chroma(
    persist_directory="./agent_memory",
    embedding_function=BaiLianEmbeddings()
)

# 2. 新增：LangGraph 官方记忆（替代 ConversationBufferMemory）
memory = MemorySaver()

# ====================== 系统提示词（极简，省Token，主功能优先） ======================
SYSTEM_PROMPT = """
你是专注于【梦境图片生成】的智能助手，天气查询是附加功能。
规则：
1. 附加：可查询城市天气
2. 必须返回【结构化数据】，禁止自然语言废话
3. 严格遵守调用限制，不浪费Token
4. 必须询问/提取用户上传的【本地图片路径】
5. 把图片路径传给 generate_dream 函数
6. 生成后，在结构化响应中返回原图+生成图路径
7. 无图片路径时，在结构化响应中提示用户上传图片
"""


# ====================== 🔥 官方规范：创建 Agent（结构化输出 + 错误处理） ======================
model = BaiLianChatModel()
agent = create_agent(
    model=model,
    tools=TOOLS,
    middleware=MIDDLEWARES
)


# ====================== 🔥 Web GUI 启动（用户交互界面） ======================
def agent_gui_chat(user_text: str, user_image):
    """
    GUI 交互函数：接收用户文字 + 上传的图片 → 调用 Agent → 流式返回结果
    完全复用你原有的 agent.stream 逻辑
    """
    if user_image is not None:
        img_path = "user_uploaded_image.png"
        # ✅ 唯一正确写法：从 namespace 中取出真实的 PIL 图片
        pil_img = user_image.image
        pil_img.save(img_path)
        
        user_input = {
            "messages": [{"role": "user", "content": f"{user_text} 图片路径：{img_path}"}]
        }
    else:
        user_input = {
            "messages": [{"role": "user", "content": user_text}]
        }


    # 🔥 完全复用你原有的官方流式传输逻辑
    response_text = ""
    config = {"configurable": {"thread_id": "dream_agent_001"}}
    # 在 agent.stream 上方添加这行（编译 agent，传入记忆）
    from langgraph.prebuilt import create_react_agent
    compiled_agent = agent.compile(checkpointer=memory)
    for chunk in compiled_agent.stream(user_input, config=config, stream_mode="updates"):
        for step, data in chunk.items():
            if "structured_response" in data:
                res = data["structured_response"]
                # 梦境结果：拼接图片路径
                if hasattr(res, "response_type") and res.response_type == "dream":
                    if res.success:
                        response_text += f"\n🖼️ 你的图片：{res.user_uploaded_image}"
                        response_text += f"\n✨ 梦境图片：{res.output_image}"
                    else:
                        response_text += f"\n{res.punny_response}"
                # 天气结果
                elif hasattr(res, "response_type") and res.response_type == "weather":
                    response_text += f"\n{res.punny_response}\n{res.weather_info}"
            
            # 流式输出打字效果
            yield response_text.strip()

# ====================== 启动 Web 界面 ======================
if __name__ == "__main__":
    with gr.Blocks(title="梦境生成 Agent") as demo:
        gr.Markdown("# 🌙 图片生成梦境智能助手")
        gr.Markdown("核心功能：上传图片 → 生成艺术梦境 | 附加：天气查询")
        
        # 输入区域
        with gr.Row():
            user_text = gr.Textbox(label="输入指令（例：帮我生成甜蜜梦境）", placeholder="我想做个好梦~")
            user_image = gr.Image(label="上传图片（可选）", type="pil", height=200)
        
        # 输出区域
        output = gr.Textbox(label="AI 响应（流式实时输出）", lines=8)
        
        # 按钮
        submit_btn = gr.Button("🚀 开始生成", variant="primary")
        
        # 绑定事件（流式输出）
        submit_btn.click(
            fn=agent_gui_chat,
            inputs=[user_text, user_image],
            outputs=output,
        )
    
    # 启动本地网页
    demo.launch(server_name="0.0.0.0", server_port=7860, show_error=True, inbrowser=True)
